# Rapport de Pipeline – Visualisations & Contrôles (v4, avec AVANT/APRÈS)

## Plan (mis à jour)
1. **Configuration et Imports**
2. **Analyse des Données Textuelles**
3. **Analyse du Prétraitement Texte**
4. **Analyse des Images**
5. **Impact de la Réduction de Dimension**
6. **Performance du Modèle Final**



# --- SETUP ALIAS DE CHEMIN + HELPERS PLOTS ---

1. **Configuration et Imports**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
from sklearn.metrics import confusion_matrix, classification_report
import plotly.express as px
from wordcloud import WordCloud



[INFO] PROJECT_ROOT = c:\Users\julie\Dropbox\10 Learning\01 LES MINES DATASCIENTEST\Rakuten\jul25_bootcamp_ds_classification-de-produits-e--commerce-rakuten-main
[INFO] FIG_DIR = c:\Users\julie\Dropbox\10 Learning\01 LES MINES DATASCIENTEST\Rakuten\jul25_bootcamp_ds_classification-de-produits-e--commerce-rakuten-main\reports\figs


In [ ]:
# Configuration générale
%matplotlib inline
plt.style.use('seaborn')
sns.set_theme(style="whitegrid")

# Logging configuration
import logging
logging.basicConfig(level=logging.INFO)

2. **Analyse des Données Textuelles**


In [ ]:

# Chargement des données
X_train = pd.read_csv("data/X_train_update.csv")
y_train = pd.read_csv("data/Y_train_CVw08PX.csv")

# Distribution des classes
plt.figure(figsize=(15, 6))
sns.countplot(data=y_train, x='prdtypecode')
plt.title('Distribution des Classes')
plt.xticks(rotation=45)

3. **Analyse du Prétraitement Texte**

In [ ]:


from features.text_cleaner import clean_text
from features.text_vectorizer import TextVectorizer

# Exemple avant/après nettoyage
sample_texts = X_train['designation'].head(5)
cleaned_texts = [clean_text(text) for text in sample_texts]

# Afficher comparaison
for orig, cleaned in zip(sample_texts, cleaned_texts):
    print(f"Original: {orig}\nNettoyé: {cleaned}\n")

# TF-IDF Analysis
vectorizer = TextVectorizer()
X_tfidf = vectorizer.fit_transform(cleaned_texts)


4. **Analyse des Images**

In [ ]:


from features.image_loader import ImageLoader
from features.image_stats import ImageStatsFeaturizer

# Charger quelques images d'exemple
loader = ImageLoader("data/images/images/image_train")
sample_images = loader.transform(X_train.head(9))

# Afficher grille d'images
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for idx, ax in enumerate(axes.flat):
    ax.imshow(sample_images[idx])
    ax.axis('off')

5. **Impact de la Réduction de Dimension**

In [ ]:



from sklearn.decomposition import PCA, TruncatedSVD

# Comparer PCA vs SVD
methods = {
    'PCA': PCA(n_components=0.90),
    'TruncatedSVD': TruncatedSVD(n_components=100)
}

results = {}
for name, method in methods.items():
    transformed = method.fit_transform(sample_images.reshape(len(sample_images), -1))
    results[name] = {
        'shape': transformed.shape,
        'explained_variance': method.explained_variance_ratio_.sum()
    }

6. **Performance du Modèle Final**

In [ ]:


# Charger résultats
results_df = pd.read_csv("models/compare_cv_results.csv")

# Plot performances par classe
plt.figure(figsize=(15, 6))
sns.barplot(data=results_df, x='class', y='f1_score')
plt.title('F1-Score par Classe')
plt.xticks(rotation=45)